<a href="https://colab.research.google.com/github/AzenSajid/FYDP-2026-27/blob/main/FYDP_2026_27.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# CELL 1: ENVIRONMENT SETUP & LIBRARIES

#basic librabries
import os
import sys
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

# 1. Ensure Reproducibility (Crucial for tracking model performance changes)
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

# 2. Hardware Allocation Check as we need max GPU and CPU power

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"--> Using compute device: {device}")
if torch.cuda.is_available():
    print(f"--> GPU Hardware: {torch.cuda.get_device_name(0)}")
else:
    print("--> WARNING: Running on CPU! Go to Runtime > Change runtime type to select a GPU.")


--> Using compute device: cpu
--> WARNING: Running on CPU! Go to Runtime > Change runtime type to select a GPU.


In [8]:
# CELL 2: DATASET HIERARCHY SETUP (UPDATED)

import os

# Your exact path from the screenshot
base_path = '/content/drive/MyDrive/ColabNotebooks/FYDP2026-27'
splits = ['train', 'validate', 'test']

print(f"Checking dataset hierarchy at: {base_path}\n")

for split in splits:
    # Create the images directory inside train/validate/test
    split_img_dir = os.path.join(base_path, split, 'images')
    os.makedirs(split_img_dir, exist_ok=True)

    # Initialize a blank CSV file for labels if it doesn't exist yet
    csv_file_path = os.path.join(base_path, split, 'labels.csv')
    if not os.path.exists(csv_file_path):
        with open(csv_file_path, 'w') as f:
            f.write("id_code,diagnosis\n")

    print(f"📂 {split}/")
    print(f" ┣━━ 📂 images/")
    print(f" ┗━━ 📄 labels.csv")

print("\n✅ Dataset hierarchy is fully set up and ready!")

Checking dataset hierarchy at: /content/drive/MyDrive/ColabNotebooks/FYDP2026-27

📂 train/
 ┣━━ 📂 images/
 ┗━━ 📄 labels.csv
📂 validate/
 ┣━━ 📂 images/
 ┗━━ 📄 labels.csv
📂 test/
 ┣━━ 📂 images/
 ┗━━ 📄 labels.csv

✅ Dataset hierarchy is fully set up and ready!


In [9]:
# CELL 3: FROZEN RESNET-50 BACKBONE (BASELINE)

import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

class BaselineResNet50(nn.Module):
    def __init__(self, num_classes=5):
        super(BaselineResNet50, self).__init__()

        # 1. Load the pre-trained ResNet-50 engine
        # Matches the GitHub repo's strategy of using ImageNet weights
        base_model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

        # 2. Isolate the feature extractor
        # Strips the final layer (matches GitHub's `include_top=False` parameter)
        self.feature_extractor = nn.Sequential(*list(base_model.children())[:-1])

        # 3. FREEZE THE BACKBONE (Crucial for Continual Learning)
        # We lock these weights so the model doesn't forget foundational visual features
        for param in self.feature_extractor.parameters():
            param.requires_grad = False

        # 4. Standard Classification Head for 0-4 DR severity
        # (We will upgrade these static Linear layers to LoRA matrices in the next cell)
        num_features = base_model.fc.in_features # ResNet-50 outputs 2048 features

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(num_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        # The raw image passes through the frozen backbone
        features = self.feature_extractor(x)
        # The extracted features pass through the classifier
        predictions = self.classifier(features)
        return predictions

# --- ARCHITECTURE AND SHAPE TEST ---
if __name__ == "__main__":
    print("Building Frozen ResNet-50 Backbone...")
    model = BaselineResNet50(num_classes=5)

    # Verify the freezing process
    frozen_params = sum(p.numel() for p in model.feature_extractor.parameters() if not p.requires_grad)
    print(f"Locked parameters in backbone: {frozen_params:,}")

    # Simulate a batch of 4 retinal images (3 color channels, 224x224 pixels)
    dummy_input = torch.randn(4, 3, 224, 224)

    try:
        dummy_output = model(dummy_input)
        print(f"✅ Success! Network Output Shape: {dummy_output.shape}")
        print("Meaning: 4 input images successfully produced 5 DR class probabilities each.")
    except Exception as e:
        print(f"❌ Architecture Error: {e}")

Building Frozen ResNet-50 Backbone...
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 114MB/s]


Locked parameters in backbone: 23,508,032
✅ Success! Network Output Shape: torch.Size([4, 5])
Meaning: 4 input images successfully produced 5 DR class probabilities each.


In [10]:
# CELL 4: LoRA MODULE & DANN DOMAIN ADAPTATION HEAD

import math
import torch
import torch.nn as nn
import torch.autograd as autograd

# 1. Gradient Reversal Layer (GRL) for Domain Adaptation (DANN)
class GradientReversalLayer(autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        # Reverses the gradient and multiplies by alpha to confuse the domain classifier
        return grad_output.neg() * ctx.alpha, None

def grad_reverse(x, alpha=1.0):
    return GradientReversalLayer.apply(x, alpha)


# 2. Low-Rank Adaptation (LoRA) Linear Layer for Parameter-Efficient Updates
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank=4, alpha=8):
        super(LoRALinear, self).__init__()
        # Base pretrained linear layer (frozen)
        self.pretrained = nn.Linear(in_features, out_features)
        self.pretrained.weight.requires_grad = False
        self.pretrained.bias.requires_grad = False

        # Trainable low-rank decomposition matrices (A and B)
        self.lora_A = nn.Parameter(torch.zeros(in_features, rank))
        self.lora_B = nn.Parameter(torch.zeros(rank, out_features))
        self.scaling = alpha / rank

        # Initialize matrices
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B) # Starts at zero so initial model behavior matches pretrained baseline

    def forward(self, x):
        base_out = self.pretrained(x)
        lora_out = (x @ self.lora_A @ self.lora_B) * self.scaling
        return base_out + lora_out


# 3. Complete Continual DR Model (Frozen ResNet50 + LoRA DR Head + DANN Domain Head)
class ContinualDRModel(nn.Module):
    def __init__(self, base_resnet_model, num_classes=5, num_domains=2, lora_rank=8):
        super(ContinualDRModel, self).__init__()

        # Reuse the frozen feature extractor from our previous cell
        self.feature_extractor = base_resnet_model.feature_extractor
        num_features = 2048 # ResNet-50 feature dimension

        # Head A: DR Severity Classification (Using LoRA for parameter efficiency)
        self.dr_classifier = nn.Sequential(
            LoRALinear(num_features, 512, rank=lora_rank),
            nn.ReLU(),
            nn.Dropout(0.3),
            LoRALinear(512, num_classes, rank=lora_rank) # Outputs 0-4 DR severity
        )

        # Head B: Domain Adversarial Classifier (Using LoRA to handle clinic/scanner shifts)
        self.domain_classifier = nn.Sequential(
            LoRALinear(num_features, 512, rank=lora_rank),
            nn.ReLU(),
            nn.Dropout(0.3),
            LoRALinear(512, num_domains, rank=lora_rank) # Outputs Source vs Target domain
        )

    def forward(self, x, alpha=1.0):
        # 1. Extract shared visual features through the frozen backbone
        features = self.feature_extractor(x)
        features = features.view(features.size(0), -1) # Flatten to batch_size x 2048

        # 2. Main Task: Predict DR Severity
        dr_preds = self.dr_classifier(features)

        # 3. Adversarial Task: Predict Domain (with Gradient Reversal)
        reversed_features = grad_reverse(features, alpha)
        domain_preds = self.domain_classifier(reversed_features)

        return dr_preds, domain_preds

# --- ARCHITECTURE TEST ---
if __name__ == "__main__":
    print("Integrating LoRA and DANN modules with ResNet-50...")
    # Pass our existing frozen model instance into the new continual architecture
    continual_model = ContinualDRModel(base_resnet_model=model)

    # Test with dummy batch of 4 images
    dummy_images = torch.randn(4, 3, 224, 224)
    dr_out, domain_out = continual_model(dummy_images)

    print(f"✅ Success! DR Output Shape: {dr_out.shape} (Expected: [4, 5])")
    print(f"✅ Success! Domain Output Shape: {domain_out.shape} (Expected: [4, 2])")
    print("Your full Continual Learning (ResNet50 + DANN + LoRA) architecture is ready!")

Integrating LoRA and DANN modules with ResNet-50...
✅ Success! DR Output Shape: torch.Size([4, 5]) (Expected: [4, 5])
✅ Success! Domain Output Shape: torch.Size([4, 2]) (Expected: [4, 2])
Your full Continual Learning (ResNet50 + DANN + LoRA) architecture is ready!
